In [2]:
!pip install 'powerlaw'

In [3]:
import bmll2 as b2
from bmll2 import reference, Security, NormalisedSecurity, SparkHelper, get_market_data, get_market_data_range, VenueMarketError, get_market_tables, save_spark_dataframe, load_spark_dataframe

b2.get_file('modules/metaorder.py')
b2.get_file('modules/LMF.py')

import metaorder as metaorder
import LMF as LMF

import random
import math
import pandas as pd
import polars as pl
import numpy as np
from pandas import StringDtype
from datetime import timedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import LogFormatterSciNotation

from statsmodels.sandbox.stats.runs import runstest_1samp 
import powerlaw
import itertools
import pylab
import scipy.stats
import scipy.fft
from scipy.optimize import curve_fit
from sklearn.model_selection import ParameterGrid

from statsmodels.tsa.stattools import acf
from scipy.stats import t
from scipy.special import gamma as gamma_func

In [4]:
trader_participation_distribution = 'power'
group                             = 'year'
config                            = 'metaorder'

In [5]:

if trader_participation_distribution == 'power':
    param_grid = {'N': [5, 10, 20, 30, 40, 50, 100, 500, 1000, 1500], 'delta': [1.5, 2, 3, 4, 5]}
if trader_participation_distribution == 'homogenous':
    param_grid = {'N': [5, 10, 20, 30, 40, 50, 100, 500, 1000, 1500]}

expanded_grid = list(ParameterGrid(param_grid))
print(expanded_grid)

[{'N': 5, 'delta': 1.5}, {'N': 5, 'delta': 2}, {'N': 5, 'delta': 3}, {'N': 5, 'delta': 4}, {'N': 5, 'delta': 5}, {'N': 10, 'delta': 1.5}, {'N': 10, 'delta': 2}, {'N': 10, 'delta': 3}, {'N': 10, 'delta': 4}, {'N': 10, 'delta': 5}, {'N': 20, 'delta': 1.5}, {'N': 20, 'delta': 2}, {'N': 20, 'delta': 3}, {'N': 20, 'delta': 4}, {'N': 20, 'delta': 5}, {'N': 30, 'delta': 1.5}, {'N': 30, 'delta': 2}, {'N': 30, 'delta': 3}, {'N': 30, 'delta': 4}, {'N': 30, 'delta': 5}, {'N': 40, 'delta': 1.5}, {'N': 40, 'delta': 2}, {'N': 40, 'delta': 3}, {'N': 40, 'delta': 4}, {'N': 40, 'delta': 5}, {'N': 50, 'delta': 1.5}, {'N': 50, 'delta': 2}, {'N': 50, 'delta': 3}, {'N': 50, 'delta': 4}, {'N': 50, 'delta': 5}, {'N': 100, 'delta': 1.5}, {'N': 100, 'delta': 2}, {'N': 100, 'delta': 3}, {'N': 100, 'delta': 4}, {'N': 100, 'delta': 5}, {'N': 500, 'delta': 1.5}, {'N': 500, 'delta': 2}, {'N': 500, 'delta': 3}, {'N': 500, 'delta': 4}, {'N': 500, 'delta': 5}, {'N': 1000, 'delta': 1.5}, {'N': 1000, 'delta': 2}, {'N': 

In [6]:
b2.get_file('top_250.csv')
top_250 = pd.read_csv('top_250.csv')

top_250_tickers = top_250['Ticker'].tolist()

In [7]:
%%time
files = b2.list_files(path = 'data/trades/top_250')

for f in files:
    b2.get_file(f'data/trades/top_250/{f}')

def load_stock(parquet):
    try:
        df = pl.read_parquet(parquet)
        return df.sort(['DateTime', 'ExchangeSequenceNo'])
    except FileNotFoundError:
        return None

exchange = 'XJSE'
stocks = {
    ticker: df
    for ticker in top_250_tickers
    if (df := load_stock(f'{ticker}_{exchange}.parquet')) is not None
}

CPU times: user 1min 4s, sys: 27.2 s, total: 1min 31s
Wall time: 1min 14s


In [7]:
trader_participation_distribution = 'power'

b2.create_folder('data', ensure_exists = True)
b2.create_folder('data/grid_search_9', ensure_exists = True)
b2.create_folder(f'data/grid_search_9/{exchange}', ensure_exists = True)
b2.create_folder(f'data/grid_search_9/{exchange}/{trader_participation_distribution}', ensure_exists = True)

In [ ]:
%time

start_from = 'BHG'
start = False

counter = 1
for ticker, stock_data in stocks.items():

    if ticker == start_from:
        start = True

    if not start:
        continue
        
    print(f'{counter}: {ticker}')
    counter += 1

    results_list = []
    for params in expanded_grid:
        
        N     = params['N']
        delta = params['delta']
        #print(f'{N} {delta}')
    
        res = metaorder.metaorder_stylised_facts(stock_data, N, delta = delta, 
                                                  trader_participation_distribution = trader_participation_distribution)
    
        if not res.is_empty():
            results_list.append(res)

    if results_list:
        results_df = pl.concat(results_list, how = 'vertical', rechunk = True)
        results_df.write_parquet(f'{ticker}_{exchange}_{trader_participation_distribution}_grid_search.parquet')
        b2.put_file(f'{ticker}_{exchange}_{trader_participation_distribution}_grid_search.parquet', f'data/grid_search_9/{exchange}/{trader_participation_distribution}')
    else:
        results_df = pl.DataFrame()
        

In [11]:
exchange = 'XJSE'
trader_participation_distribution = 'power'

In [12]:
files = b2.list_files(path = f'data/grid_search_9/{exchange}/{trader_participation_distribution}')

for f in files:
    b2.get_file(f'data/grid_search_9/{exchange}/{trader_participation_distribution}/{f}')


In [13]:
grid_search_data = pl.scan_parquet(f'*{trader_participation_distribution}_grid_search.parquet').collect()
grid_search_data = grid_search_data.sort(['ticker', 'year', 'N', 'delta'])
grid_search_data.write_csv('calibration_results.csv')
b2.put_file('calibration_results.csv', f'data/grid_search_9/{exchange}')